In [1]:
import pandas as pd
import numpy as np
import re


# =========================================================
# 1. Load Task 2 output
# =========================================================

data = pd.read_csv("cleaned_womens_clothing.csv")

print("Original Shape:", data.shape)


# =========================================================
# 2. Dynamic Parser
# =========================================================

def standardize_and_convert(df):

    data = df.copy()
    conversion_log = []

    # -----------------------------------------------------
    # A. Standardize column names
    # -----------------------------------------------------

    original_columns = data.columns.tolist()

    new_columns = []

    for column in data.columns:

        # Convert to string
        column = str(column)

        # Remove leading/trailing spaces
        column = column.strip()

        # Convert to lowercase
        column = column.lower()

        # Replace spaces and special characters with _
        column = re.sub(r"[^a-z0-9]+", "_", column)

        # Remove extra underscores
        column = re.sub(r"_+", "_", column)

        # Remove underscore from beginning/end
        column = column.strip("_")

        new_columns.append(column)

    data.columns = new_columns

    # Log changed column names
    for old, new in zip(original_columns, new_columns):

        if old != new:
            conversion_log.append(
                f"Column name standardized: '{old}' -> '{new}'"
            )


    # -----------------------------------------------------
    # B. Detect and convert incorrect data types
    # -----------------------------------------------------

    for column in data.columns:

        original_dtype = data[column].dtype

        # -------------------------------------------------
        # Skip columns that are already numeric
        # -------------------------------------------------

        if pd.api.types.is_numeric_dtype(data[column]):
            continue


        # -------------------------------------------------
        # Try numeric conversion safely
        # -------------------------------------------------

        if data[column].dtype == "object":

            non_null = data[column].dropna()

            if len(non_null) == 0:
                continue

            numeric_version = pd.to_numeric(
                non_null,
                errors="coerce"
            )

            conversion_ratio = numeric_version.notna().mean()

            # Convert only when almost all non-null values
            # can safely become numeric
            if conversion_ratio >= 0.95:

                converted = pd.to_numeric(
                    data[column],
                    errors="coerce"
                )

                # Make sure conversion did not create
                # unexpected missing values
                original_non_null = data[column].notna().sum()
                converted_non_null = converted.notna().sum()

                if original_non_null == converted_non_null:

                    data[column] = converted

                    # Convert integer-like floats to Int64
                    if (
                        pd.api.types.is_float_dtype(data[column])
                        and (data[column].dropna() % 1 == 0).all()
                    ):
                        data[column] = data[column].astype("Int64")

                    conversion_log.append(
                        f"Data type converted: "
                        f"'{column}' : {original_dtype} -> "
                        f"{data[column].dtype}"
                    )

                    continue


        # -------------------------------------------------
        # Try boolean detection
        # -------------------------------------------------

        if data[column].dtype == "object":

            values = (
                data[column]
                .dropna()
                .astype(str)
                .str.strip()
                .str.lower()
                .unique()
            )

            boolean_values = {
                "true", "false",
                "yes", "no"
            }

            if len(values) > 0 and set(values).issubset(boolean_values):

                data[column] = (
                    data[column]
                    .astype(str)
                    .str.strip()
                    .str.lower()
                    .map({
                        "true": True,
                        "false": False,
                        "yes": True,
                        "no": False
                    })
                )

                conversion_log.append(
                    f"Data type converted: "
                    f"'{column}' : {original_dtype} -> bool"
                )


    # -----------------------------------------------------
    # C. Final pandas type optimization
    # -----------------------------------------------------

    data = data.convert_dtypes()


    # -----------------------------------------------------
    # D. Final summary
    # -----------------------------------------------------

    conversion_log.append(
        f"Final shape: {data.shape[0]} rows, "
        f"{data.shape[1]} columns"
    )

    return data, conversion_log


# =========================================================
# 3. Run Parser
# =========================================================

data_clean, conversion_log = standardize_and_convert(data)


# =========================================================
# 4. Display Cleaning Log
# =========================================================

print("\nData Standardization Log")
print("-" * 50)

for log in conversion_log:
    print(log)


# =========================================================
# 5. Check Final Column Names
# =========================================================

print("\nFinal Column Names:")
print(data_clean.columns.tolist())


# =========================================================
# 6. Check Final Data Types
# =========================================================

print("\nFinal Data Types:")
print(data_clean.dtypes)


# =========================================================
# 7. Verify Dataset
# =========================================================

print("\nFinal Shape:", data_clean.shape)

print("\nMissing Values:")
print(data_clean.isna().sum())

print("\nFirst 5 Rows:")
print(data_clean.head())


# =========================================================
# 8. Save Final Dataset
# =========================================================

data_clean.to_csv(
    "standardized_womens_clothing.csv",
    index=False
)

print(
    "\nStandardized dataset saved successfully."
)

Original Shape: (23486, 10)

Data Standardization Log
--------------------------------------------------
Column name standardized: 'Clothing ID' -> 'clothing_id'
Column name standardized: 'Age' -> 'age'
Column name standardized: 'Title' -> 'title'
Column name standardized: 'Review Text' -> 'review_text'
Column name standardized: 'Rating' -> 'rating'
Column name standardized: 'Recommended IND' -> 'recommended_ind'
Column name standardized: 'Positive Feedback Count' -> 'positive_feedback_count'
Column name standardized: 'Division Name' -> 'division_name'
Column name standardized: 'Department Name' -> 'department_name'
Column name standardized: 'Class Name' -> 'class_name'
Final shape: 23486 rows, 10 columns

Final Column Names:
['clothing_id', 'age', 'title', 'review_text', 'rating', 'recommended_ind', 'positive_feedback_count', 'division_name', 'department_name', 'class_name']

Final Data Types:
clothing_id                         Int64
age                                 Int64
title   